# Third LAB

### Select the Kernel

Click on "Select Kernel" on the Top Right

Choose "Python Environments..."

Then choose the one that looks like `.venv (Python 3.12.x) .venv/bin/python` - it should be marked as "Recommended" and have a big star next to it.

In [1]:
# imports

import os
from dotenv import load_dotenv
# from scraper import *
from IPython.display import Markdown, display
from huggingface_hub import login, InferenceClient
from tqdm.notebook import tqdm
from openai import OpenAI
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc
# import argparse
# import requests
# import re
# import json
# from bs4 import BeautifulSoup

In [2]:
# Check for GPU / CPU
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = "cuda" if torch.cuda.is_available() else "cpu"

GPU: NVIDIA GeForce RTX 3050 Laptop GPU
CUDA: 12.1


### Connecting to Hugging Face

The next cell is where we load in the environment variables in your `.env` file and connect to Hugging Face.  

In [3]:
# Log in to HuggingFace - if you get a "Note" about Environment variable being set, ignore it

hf_token = os.environ['HF_TOKEN']
client = InferenceClient(api_key=hf_token) 
login(hf_token, add_to_git_credential = False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Function to call Models

In [4]:
def model_call(hf_token, model_id, prompt):

    # This client sends requests to a remote inference provider.
    client = InferenceClient(
        api_key=hf_token,
        provider="auto"
    )

    # Load model directly
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Load model directly onto available hardware
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=device,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    )

    print("Model loaded successfully!")
    print("Memory footprint:", round(model.get_memory_footprint() / 1024**3, 2), "GB")

    
    # Defining prompt & message
    messages=[
            {"role": "user", "content": prompt}
        ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    # Output
    outputs = model.generate(**inputs, max_new_tokens=250)

    # Extract Response
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:], 
        clean_up_tokenization_spaces=False)

    # Free up memory in GPU 
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    
    return print("Answer to the puzzle:", response)

### Testing out SLMs

In [5]:
# Define Puzzle
puzzle = """
Three identical books are placed side by side on a shelf in the following order:
- Book A: 300 pages
- Book B: 400 pages
- Book C: 500 pages
Each book has a front and back cover, each exactly 3 mm thick. The pages themselves are 0.1 mm thick.

A tiny worm starts at the first page of Book A and eats its way in a straight line, perpendicular to the pages, until it reaches the last page of Book C.

Assume the books are standing upright, with their spines facing outward and their front covers facing the same direction.

What is the total distance the worm travels through the books? Give the answer without any explanation.
"""

In [6]:
# Llama Model
model_call(hf_token, "meta-llama/Llama-3.2-3B-Instruct", puzzle)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Model loaded successfully!
Memory footprint: 5.98 GB
Answer to the puzzle: 2.1 meters.<|eot_id|>


In [7]:
# Qwen Model
model_call(hf_token, "Qwen/Qwen2.5-3B-Instruct", puzzle)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Memory footprint: 5.75 GB
Answer to the puzzle: 2997 mm<|im_end|>


In [9]:
# Phi Model
model_call(hf_token, "microsoft/Phi-4-mini-instruct", puzzle)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!
Memory footprint: 7.15 GB
Answer to the puzzle: The total distance the worm travels through the books is 3.6 mm.<|end|>


In [8]:
# Gemma Model
model_call(hf_token, "google/gemma-3-1b-it", puzzle)

Model loaded successfully!
Memory footprint: 1.86 GB
Answer to the puzzle: 300 mm + 400 mm + 500 mm = 1200 mm

The total distance the worm travels is 1200 mm.

Final Answer: The final answer is 1200<end_of_turn>
